# Lab 09 - Mạng nơ-ron nhiều lớp (MLP)

Sinh viên: **Lê Quyết Tiến** - MSSV: **2474802010386**.

Notebook chạy từ thư mục gốc `09. MLP`. Trên Colab, hãy mount Google Drive, `%cd` đến thư mục này rồi chọn **Runtime > Run all**. Giữ nguyên các tệp CSV trong `data/`.


## Bước 1 - Tự tính lan truyền tiến


In [1]:
# -*- coding: utf-8 -*-
"""Bước 1: tự tính lan truyền tiến bằng numpy."""

import numpy as np


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def relu(z):
    return np.maximum(0, z)


w = np.array([0.8, -0.5, 0.3])
b = -0.2
x = np.array([2.0, 1.0, 3.0])
z = w @ x + b
print("Mot no-ron:")
print(f" z = {w[0]}*{x[0]} + ({w[1]})*{x[1]} + {w[2]}*{x[2]} + ({b}) = {z:.2f}")
print(f" a = sigmoid(z) = {sigmoid(z):.4f}")
print()

W1 = np.array([[1.0, 1.0], [1.0, 1.0]])
b1 = np.array([0.0, -1.0])
W2 = np.array([1.0, -2.0])
b2 = 0.0
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
print("Mang giai XOR:")
print(" x1 x2 h1 h2 y")
for x in X:
    h = relu(x @ W1 + b1)
    y = h @ W2 + b2
    print(f" {x[0]:.0f}  {x[1]:.0f}  {h[0]:.0f}  {h[1]:.0f}  {y:.0f}")
print()
H = relu(X @ W1 + b1)
Y = H @ W2 + b2
print("Tinh ca bang mot lan:", Y)


Mot no-ron:
 z = 0.8*2.0 + (-0.5)*1.0 + 0.3*3.0 + (-0.2) = 1.80
 a = sigmoid(z) = 0.8581

Mang giai XOR:
 x1 x2 h1 h2 y
 0  0  0  0  0
 0  1  1  0  1
 1  0  1  0  1
 1  1  2  1  0

Tinh ca bang mot lan: [0. 1. 1. 0.]


## Bước 2 - Một nơ-ron và một lớp ẩn trên XOR


In [2]:
# -*- coding: utf-8 -*-
"""Bước 2: một nơ-ron không giải được XOR, một lớp ẩn thì được."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

df = pd.read_csv("data/xor.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

lr = LogisticRegression().fit(X_tr, y_tr)
print(f"Hoi quy logistic: {accuracy_score(y_te, lr.predict(X_te)):.4f}")
print()
print("So no-ron an  so tham so  tap hoc  tap kiem tra")
for so in [1, 2, 4, 8]:
    mlp = MLPClassifier(
        hidden_layer_sizes=(so,), solver="lbfgs", max_iter=5000, random_state=0
    )
    mlp.fit(X_tr, y_tr)
    so_ts = sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
    a_hoc = accuracy_score(y_tr, mlp.predict(X_tr))
    a_kt = accuracy_score(y_te, mlp.predict(X_te))
    print(f" {so:2d}           {so_ts:3d}     {a_hoc:.4f}       {a_kt:.4f}")


Hoi quy logistic: 0.4500

So no-ron an  so tham so  tap hoc  tap kiem tra
  1             5     0.6533       0.6900
  2             9     0.6533       0.6900
  4            17     0.9033       0.9300
  8            33     0.9167       0.9100


## Bước 3 - Độ lớn mạng và điều chuẩn


In [3]:
# -*- coding: utf-8 -*-
"""Bước 3: mạng to thì học vẹt, điều chuẩn bằng alpha thì đỡ."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)


def mang(lop_an, alpha):
    return make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=lop_an,
            alpha=alpha,
            solver="lbfgs",
            max_iter=20000,
            random_state=0,
        ),
    )


print("Ba mang                     tap hoc  tap kiem tra")
for ten, lop_an, alpha in [
    ("(2,), alpha 1e-4", (2,), 1e-4),
    ("(100, 100), alpha 1e-5", (100, 100), 1e-5),
    ("(100, 100), alpha 1.0", (100, 100), 1.0),
]:
    m = mang(lop_an, alpha).fit(X_tr, y_tr)
    print(
        f" {ten:27s} {accuracy_score(y_tr, m.predict(X_tr)):.4f}"
        f"       {accuracy_score(y_te, m.predict(X_te)):.4f}"
    )
print()

print("alpha       kiem dinh cheo")
diem = {}
for alpha in [1e-5, 1e-3, 0.1, 1.0, 3.0, 10.0]:
    diem[alpha] = cross_val_score(mang((100, 100), alpha), X_tr, y_tr, cv=5).mean()
    print(f" {alpha:<10g} {diem[alpha]:.4f}")
a_tot = max(diem, key=diem.get)
m = mang((100, 100), a_tot).fit(X_tr, y_tr)
print(f"alpha tot nhat = {a_tot:g}, tap kiem tra {accuracy_score(y_te, m.predict(X_te)):.4f}")


Ba mang                     tap hoc  tap kiem tra
 (2,), alpha 1e-4            0.8444       0.8533
 (100, 100), alpha 1e-5      1.0000       0.7867
 (100, 100), alpha 1.0       0.8800       0.9067

alpha       kiem dinh cheo
 1e-05      0.7911
 0.001      0.7867
 0.1        0.7556
 1          0.8578
 3          0.8444
 10         0.8356
alpha tot nhat = 1, tap kiem tra 0.9067


## Bước 4 - Mạng nơ-ron khi quên chuẩn hóa


In [4]:
# -*- coding: utf-8 -*-
"""Bước 4: mạng nơ-ron khi quên chuẩn hoá."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_mo_rong.csv")
X = df[["gio_on", "diem_giua_ky", "chi_tieu"]]
y = df["qua_mon"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

tho = MLPClassifier(
    hidden_layer_sizes=(20,), max_iter=3000, random_state=0
).fit(X_tr, y_tr)
chuan = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(20,), max_iter=3000, random_state=0),
).fit(X_tr, y_tr)

print(f"Doan bua ca lop qua : {y_te.mean():.4f}")
print(f"Khong chuan hoa     : {accuracy_score(y_te, tho.predict(X_te)):.4f}")
print(f"Co chuan hoa        : {accuracy_score(y_te, chuan.predict(X_te)):.4f}")
print()
du = tho.predict(X_te)
print("Khong chuan hoa, so ban doan qua:", du.sum(), "tren", len(du))


Doan bua ca lop qua : 0.6000
Khong chuan hoa     : 0.4000
Co chuan hoa        : 0.9000

Khong chuan hoa, so ban doan qua: 0 tren 30


## Bước 5 - Nhận dạng chữ số viết tay


In [5]:
# -*- coding: utf-8 -*-
"""Bước 5: nhận dạng chữ số viết tay bằng MLP."""

import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

mang = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=0),
).fit(X_tr, y_tr)
mlp = mang[-1]
print("Kich thuoc cac ma tran trong so:", [w.shape for w in mlp.coefs_])
so_ts = sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
print("Tong so tham so:", so_ts)
print("So vong hoc:", mlp.n_iter_)
mm = mlp.loss_curve_
print(f"Mat mat vong 1: {mm[0]:.4f}, vong 10: {mm[9]:.4f}, vong cuoi: {mm[-1]:.4f}")
print()

du = mang.predict(X_te)
print(f"Accuracy tren tap kiem tra: {accuracy_score(y_te, du):.4f}")
sai = np.flatnonzero(du != y_te)
print("So anh doan sai:", len(sai), "tren", len(y_te))
for i in sai:
    print(f" anh {i:3d}: that la {y_te[i]}, doan la {du[i]}")
print()

dung_som = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(100,), max_iter=1000, early_stopping=True, random_state=0
    ),
).fit(X_tr, y_tr)
print(
    "Dung som sau", dung_som[-1].n_iter_, "vong, accuracy",
    f"{accuracy_score(y_te, dung_som.predict(X_te)):.4f}"
)
print()

khac = {
    "Logistic co chuan hoa": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000)
    ),
    "KNN, k = 5": KNeighborsClassifier(),
    "SVM kernel RBF": SVC(),
}
for ten, mh in khac.items():
    acc = accuracy_score(y_te, mh.fit(X_tr, y_tr).predict(X_te))
    print(f"{ten:22s} {acc:.4f}")


Kich thuoc cac ma tran trong so: [(64, 100), (100, 10)]
Tong so tham so: 7510
So vong hoc: 144
Mat mat vong 1: 2.5206, vong 10: 0.4448, vong cuoi: 0.0052

Accuracy tren tap kiem tra: 0.9844
So anh doan sai: 7 tren 450
 anh   6: that la 9, doan la 3
 anh  61: that la 5, doan la 7
 anh  77: that la 2, doan la 1
 anh  85: that la 9, doan la 7
 anh 276: that la 8, doan la 2
 anh 428: that la 7, doan la 9
 anh 443: that la 4, doan la 7

Dung som sau 44 vong, accuracy 0.9578

Logistic co chuan hoa  0.9711
KNN, k = 5             0.9867
SVM kernel RBF         0.9911
